## PENDEKATAN 3 VADER TRANSLATION BASED (VTB)

In [1]:
import sys
!{sys.executable} -m pip install deep-translator tqdm
!{sys.executable} -m pip install ipywidgets


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
import time
import warnings
from tqdm import tqdm  
from deep_translator import GoogleTranslator

warnings.filterwarnings('ignore')

# Konfigurasi path
DATA_PATH = '../../../datapreprocessingcopy/data_preprocessing_final.csv'
OUTPUT_DIR = '../../outputs/VTB'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [3]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

# Validasi kolom
required_cols = ['no', 'timestamp', 'teks', 'teks_processed']
assert all(col in df.columns for col in required_cols), f"Kolom wajib {required_cols} tidak ditemukan!"

print(f"\nData berhasil dimuat: {len(df)} tweet")
print(f"Kolom tersedia: {df.columns.tolist()}")
df.head()


Data berhasil dimuat: 13192 tweet
Kolom tersedia: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertib media online DPR pemerintah jangan spor...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus evaluasi lagi kebijakan bebas visa utama...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang un...
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,bebas suara dapat memang jamin UU tetapi bebas...


In [4]:
# 2.3 Inisialisasi Translator Google (ID -> EN)
translator = GoogleTranslator(source='id', target='en')

def translate_tweet(text):
    """
    Fungsi translasi dengan proteksi error dan mekanisme retry.
    """
    if not isinstance(text, str) or pd.isna(text) or len(text.strip()) == 0:
        return ""
    
    try:
        # Percobaan pertama
        translated = translator.translate(text[:4500])
        return translated
    except Exception as e:
        # Jika gagal (biasanya karena koneksi/limit), tunggu 5 detik lalu coba sekali lagi
        print(f"\n[RETRY] Gagal pada teks: {text[:50]}... | Error: {e}")
        print("Menunggu 5 detik sebelum mencoba lagi...")
        time.sleep(5) 
        try:
            return translator.translate(text[:4500])
        except:
            # Jika tetap gagal setelah retry, kembalikan string kosong agar pipeline tidak mati
            return "" 

print("[INFO] Fungsi translate_tweet dengan mekanisme retry berhasil didefinisikan.")

[INFO] Fungsi translate_tweet dengan mekanisme retry berhasil didefinisikan.


In [5]:
# 2.4 Jalankan Translasi dengan Progress Bar dan Proteksi Rate Limit
print("\n[PROSES] Menerjemahkan teks ke Bahasa Inggris...")
print(f"Total data: {len(df)} tweet")
print("Estimasi waktu: ~20-40 menit (tergantung limit API & stabilitas internet)")

translated_texts = []

# Menggunakan enumerate untuk melacak indeks data (i)
for i, text in enumerate(tqdm(df['teks_processed'], desc="Translating")):
    # Panggil fungsi translasi
    translated_texts.append(translate_tweet(text))
    
    # 1. Proteksi Rate Limit: Jeda 1 detik setiap 50 tweet
    if (i + 1) % 50 == 0:
        time.sleep(1)
    
    # 2. Checkpoint: Simpan hasil sementara setiap 2000 tweet
    if (i + 1) % 2000 == 0:
        temp_df = df.iloc[:len(translated_texts)].copy()
        temp_df['teks_translated'] = translated_texts
        CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, f'checkpoint_{i+1}.csv')
        temp_df.to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8')
        print(f"\n[CHECKPOINT] Berhasil menyimpan {i+1} data ke {CHECKPOINT_FILE}")

# Masukkan hasil akhir ke dataframe utama
df['teks_translated'] = translated_texts

print("\n[INFO] Seluruh proses translasi selesai.")


[PROSES] Menerjemahkan teks ke Bahasa Inggris...
Total data: 13192 tweet
Estimasi waktu: ~20-40 menit (tergantung limit API & stabilitas internet)


Translating:  15%|█▌        | 2000/13192 [48:10<5:40:19,  1.82s/it] 


[CHECKPOINT] Berhasil menyimpan 2000 data ke ../../outputs/VTB\checkpoint_2000.csv


Translating:  30%|███       | 4000/13192 [1:29:49<2:59:50,  1.17s/it]


[CHECKPOINT] Berhasil menyimpan 4000 data ke ../../outputs/VTB\checkpoint_4000.csv


...


[RETRY] Gagal pada teks: apa yang telah saya muka di atas jadi suatu benar ... | Error: apa yang telah saya muka di atas jadi suatu benar dan alas saya untuk dukungan bahas dan pengesahan RUU TPKS di sidang paripurna sehingga turut saya tidak ada alas untuk tidak dukungan RUU TPKS RUUTPKS --> No translation was found using the current translator. Try another translator?
Menunggu 5 detik sebelum mencoba lagi...


Translating:  91%|█████████▏| 12059/13192 [3:50:13<14:25,  1.31it/s]  


[RETRY] Gagal pada teks: ada kuasa pondok pesantren di bandung dan jombang ... | Error: ada kuasa pondok pesantren di bandung dan jombang pelaksanaan cabul perkosa hadap santriwati nya otoritas pemerintah dan otoritas agama di bandung terus usaha sangkal jahat itu dua parpol di DPR RI terus jegal pengesahan RUU keras seksual --> No translation was found using the current translator. Try another translator?
Menunggu 5 detik sebelum mencoba lagi...


Translating: 100%|██████████| 13192/13192 [4:11:54<00:00,  1.15s/it]  


[INFO] Seluruh proses translasi selesai.


In [6]:
# 2.5 Validasi Hasil Translasi
null_translated = df['teks_translated'].isna().sum()
empty_translated = (df['teks_translated'].str.strip() == '').sum()

print("\n[STATISTIK] Hasil Translasi:")
print(f"  - Total tweet          : {len(df)}")
print(f"  - Gagal/Null           : {null_translated}")
print(f"  - Kosong setelah translasi: {empty_translated}")
print(f"  - Berhasil diterjemahkan: {len(df) - null_translated - empty_translated}")

# Preview hasil
print("\n[PREVIEW] 3 Tweet Pertama:")
for i in range(3):
    print(f"\n🇮🇩 Processed : {df['teks_processed'].iloc[i][:80]}...")
    print(f"🇬🇧 Translated: {df['teks_translated'].iloc[i][:80]}...")


[STATISTIK] Hasil Translasi:
  - Total tweet          : 13192
  - Gagal/Null           : 0
  - Kosong setelah translasi: 1
  - Berhasil diterjemahkan: 13191

[PREVIEW] 3 Tweet Pertama:

🇮🇩 Processed : ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
🇬🇧 Translated: FAIR, for those who have state public policy, remember this! !...

🇮🇩 Processed : tertib media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada...
🇬🇧 Translated: The DPR government's orderly online media should not be sporadic, let alone sele...

🇮🇩 Processed : harus evaluasi lagi kebijakan bebas visa utama untuk negara tiongkok pak ! ! bah...
🇬🇧 Translated: You have to re-evaluate the main visa-free policy for China, sir! ! the danger o...


In [7]:
# Mencari baris dengan hasil translasi kosong
empty_mask = df['teks_translated'].isna() | (df['teks_translated'].str.strip() == '')
df_kosong = df[empty_mask]

print(f" Ditemukan {len(df_kosong)} baris kosong:\n")
for _, row in df_kosong.iterrows():
    print(f"No   : {row['no']}")
    print(f"Asli : {row['teks_processed'][:120]}...")
    print(f"Hasil: '{row['teks_translated']}'")
    print("-" * 60)

 Ditemukan 1 baris kosong:

No   : 11994
Asli : anggota komisi IX DPR RI rahmad handoyo dorong pemerintah segera tindakan cepat rubah kebijakan pada libur natal dan tah...
Hasil: ''
------------------------------------------------------------


In [8]:
# Mengisi secara manual baris yang kosong (Indeks No: 11994)
# Kita buat manual translasinya: "Member of Commission IX of the Indonesian House of Representatives Rahmad Handoyo encourages the government to take quick action to change policies on the Christmas and New Year holidays..."

df.loc[df['no'] == 11994, 'teks_translated'] = "Member of Commission IX of the House of Representatives Rahmad Handoyo encourages the government to take immediate action to change policies during the Christmas and Year-end holidays"

print("[INFO] Baris kosong telah diisi manual.")

[INFO] Baris kosong telah diisi manual.


In [ ]:
# Validasi ulang untuk mencari baris yang kosong
empty_mask = df['teks_translated'].isna() | (df['teks_translated'].str.strip() == '')
df_kosong = df[empty_mask]

print(f" Ditemukan {len(df_kosong)} baris kosong:\n")
for _, row in df_kosong.iterrows():
    print(f"No   : {row['no']}")
    print(f"Asli : {row['teks_processed'][:120]}...")
    print(f"Hasil: '{row['teks_translated']}'")
    print("-" * 60)

 Ditemukan 0 baris kosong:



In [10]:
# 2.6 Simpan Data Hasil Translasi
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'data_translated.csv')
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[OUTPUT] Data berhasil disimpan ke: {OUTPUT_FILE}")


[OUTPUT] Data berhasil disimpan ke: ../../outputs/VTB\data_translated.csv
